# V7_C_N03 — Learning Gaps to Proportionate Support

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft using synthetic data. Outputs support authorized review only.

## Decision contract
Identify curriculum domains and schools needing further diagnostic review and support. Owners: curriculum, assessment, and education authorities. Scores do not determine learner worth, promotion, sanctions, or teacher discipline automatically.

In [1]:
import numpy as np,pandas as pd
rng=np.random.default_rng(7503);n=420;df=pd.DataFrame({'school':rng.choice([f'S{i:02d}' for i in range(18)],n),'grade':rng.choice([4,5,6],n),'domain':rng.choice(['Numeracy','Reading'],n),'score':rng.normal(500,80,n),'weight':rng.uniform(.7,1.8,n),'attendance':rng.uniform(.55,1,n)});df.loc[df.attendance<.7,'score']-=35;df.head()

## Measurement contract
Scale, domain, grade, administration, accommodations, sampling, weights, linking, standard errors, and proficiency rules must be documented. Raw scores from different forms are not automatically comparable.

In [2]:
assert df.score.notna().all()
def wmean(g): return np.average(g.score,weights=g.weight)
school=df.groupby(['school','domain']).apply(wmean,include_groups=False).rename('mean_score').reset_index();print(school.head().round(1))

  school    domain  mean_score
0    S00  Numeracy       474.1
1    S00   Reading       477.1
2    S01  Numeracy       498.4
3    S01   Reading       474.4
4    S02  Numeracy       472.0


## Uncertainty before ranking
Approximate weighted standard errors show that small differences may not be meaningful. Operational work requires design-consistent variance estimation.

In [3]:
def stats(g):
 w=g.weight.to_numpy();x=g.score.to_numpy();m=np.average(x,weights=w);neff=w.sum()**2/(w**2).sum();se=np.sqrt(np.average((x-m)**2,weights=w)/neff);return pd.Series({'mean':m,'se':se,'n_eff':neff})
res=df.groupby(['school','domain']).apply(stats,include_groups=False).reset_index();res['lower']=res['mean']-1.96*res.se;res['upper']=res['mean']+1.96*res.se;print(res.head().round(1))

  school    domain   mean    se  n_eff  lower  upper
0    S00  Numeracy  474.1  21.0   13.1  432.9  515.2
1    S00   Reading  477.1  24.7   10.9  428.7  525.5
2    S01  Numeracy  498.4  24.7    7.5  450.0  546.7
3    S01   Reading  474.4  18.1   19.5  438.9  510.0
4    S02  Numeracy  472.0  20.9   10.3  431.0  513.1


## Diagnostic—not punitive—support rule
Flag a school-domain only when the upper confidence bound remains below a reviewed benchmark. Low effective sample size causes abstention.

In [4]:
benchmark=480;res['disposition']=np.where(res.n_eff<8,'ABSTAIN—MORE EVIDENCE',np.where(res.upper<benchmark,'DIAGNOSTIC REVIEW AND SUPPORT','MONITOR'));print(res.disposition.value_counts().to_string());print(res[res.disposition!='MONITOR'].head().round(1).to_string(index=False))

disposition
MONITOR                  32
ABSTAIN—MORE EVIDENCE     4
school   domain  mean   se  n_eff  lower  upper           disposition
   S01 Numeracy 498.4 24.7    7.5  450.0  546.7 ABSTAIN—MORE EVIDENCE
   S03 Numeracy 528.7 13.2    4.9  502.8  554.6 ABSTAIN—MORE EVIDENCE
   S09 Numeracy 425.9 20.9    2.9  384.9  466.9 ABSTAIN—MORE EVIDENCE
   S10 Numeracy 445.1 33.2    3.7  380.1  510.2 ABSTAIN—MORE EVIDENCE


## Context and equity
Attendance association is descriptive, not causal. Support should address barriers rather than stigmatize learners, schools, teachers, languages, disability, or communities.

In [5]:
bands=pd.cut(df.attendance,[0,.7,.85,1],labels=['low','medium','high'],include_lowest=True);context=df.assign(attendance_band=bands).groupby('attendance_band',observed=True).apply(wmean,include_groups=False);print(context.round(1).to_string())

attendance_band
low       464.7
medium    498.5
high      492.3


## Capacity-aware remediation scenario
Prioritize a limited number of school-domain reviews while preserving domain coverage. Final interventions require local diagnostic evidence.

In [6]:
eligible=res[res.disposition=='DIAGNOSTIC REVIEW AND SUPPORT'].copy();eligible['evidence_gap']=benchmark-eligible.upper;work=(eligible.sort_values(['domain','evidence_gap'],ascending=[True,False]).groupby('domain',group_keys=False).head(5));work['status']='PENDING CURRICULUM REVIEW';print(work.round(1).to_string(index=False))

Empty DataFrame
Columns: [school, domain, mean, se, n_eff, lower, upper, disposition, evidence_gap, status]
Index: []


## Exercises
1. Add plausible values or replicate weights conceptually. 2. Compare mean and proficiency reporting. 3. Add language/accommodation checks. 4. Explain why a flagged school should not be sanctioned.

## Exact solutions
1. Combine estimates across plausible values and use the survey variance method specified by the assessment design. 2. Means and threshold shares answer different questions and both have uncertainty. 3. Verify construct comparability, accessibility, sample size, and disclosure. 4. Results may reflect sampling error, intake, resources, attendance, language, accessibility, and implementation; the output is diagnostic support evidence, not proof of misconduct.

In [7]:
assert res[['lower','upper']].notna().all().all() and set(work.status)<= {'PENDING CURRICULUM REVIEW'};print('V7_C_N03_COMPLETE_EXECUTION_PASS')

V7_C_N03_COMPLETE_EXECUTION_PASS
